In [20]:
from IPython.display import display, Image
import operator
from langgraph.graph import StateGraph, START, END
from typing import Annotated, Literal, List

from typing_extensions import TypedDict
from langgraph.types import Command, interrupt

In [21]:
class State(TypedDict):
    nlist: Annotated[List[str], operator.add]

In [22]:
def node_a(state:State)->Command[Literal['b', 'c']|type(END)]:
    print('This is node[A]')
    select = state['nlist'][-1]
    if select == 'b':
        next_node = 'b'
    elif select == 'c':
        next_node = 'c'
    elif select == 'q':
        next_node = END
    else:
        admin = interrupt(f'Unexepected input : {select}')
        print(admin)
        if admin == 'continue':
            next_node = 'b'
        else:
            next_node = END
    return Command(
        update=State(nlist=[select]),
        goto=next_node
    )

In [23]:


def node_b(state:State)->State:
    print('This is node[B]')
    return State(nlist=['B'])

def node_c(state:State)->State:
    print('This is node[C]')
    return State(nlist=['C'])

def node_d(state:State)->State:
    print('This is node[D]')
    return State(nlist=['D'])

In [24]:
def conditional_edge(state:State) -> Literal["b","c", 'd']|type[END]:
    select = state['nlist'][-1]
    if select == 'b':
        return 'b'
    elif select == 'c':
        return 'c'
    elif select == 'q':
        return END
    else:
        return END

In [25]:

builder = StateGraph(State)
builder.add_node('a', node_a)
builder.add_node('b', node_b)
builder.add_node('c', node_c)
builder.add_node('d', node_d)

builder.add_edge(START, 'a')
builder.add_edge('b', END)
builder.add_edge('c', 'd')
builder.add_edge('d', END)



In [36]:
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()
config = {
    'configurable': {
        'thread_id':'1'
    }
}

In [37]:
graph = builder.compile(checkpointer=memory)

In [38]:
while True:
    user = input('b, c, or q to quit:')
    input_state = State(
        nlist=[user]
    )
    result = graph.invoke(input_state, config)
    print(result)
    
    if '__interrupt__' in result:
        print(f'interrupt result: \n{result}')
        msg = result['__interrupt__'][-1].value
        print(msg)
        human = input(f'\n{msg}: ')
        human_response = Command(
            resume=human
        )
        print(f'human_response:\n{human_response}')
        result = graph.invoke(human_response, config)
        
    if result['nlist'][-1] == 'q':
        print('quit')
        break
    # 中断恢复后，会从节点的开始继续运行

This is node[A]
{'nlist': ['s'], '__interrupt__': [Interrupt(value='Unexepected input : s', id='74efc5d8df572d8e0b6e90c3556a3101')]}
interrupt result: 
{'nlist': ['s'], '__interrupt__': [Interrupt(value='Unexepected input : s', id='74efc5d8df572d8e0b6e90c3556a3101')]}
Unexepected input : s
human_response:
Command(resume='continue')
This is node[A]
continue
This is node[B]
This is node[A]
{'nlist': ['s', 's', 'B', 'q', 'q']}
quit
